# 🖼️ Comprehensive Digital Image Processing Project
### Course: Image Processing (ELE 261)
**Faculty of Engineering – New Ismailia National University (NINU)**  
**Author**: Ammar Abdelfattah & Team  
**Focus**: Classical Spatial & Frequency Domain Enhancement, Segmentation, and Automated Feature Extraction

---

## 📌 Project Objectives
This project provides a rigorous, modular framework covering the fundamental pillars of Digital Image Processing:
1. **Color Spaces & Point Operations**: RGB, HSV, Grayscale, Gamma Correction, Contrast Stretching.
2. **Histogram Equalization**: Global Histogram Equalization vs. CLAHE.
3. **Spatial Filtering & Edge Detection**: Linear convolution, Gaussian smoothing, Median filtering, Sobel, Laplacian, and Canny edge extraction.
4. **Frequency Domain Filtering**: 2D Fast Fourier Transform (FFT), Butterworth Low-pass & High-pass filtering.
5. **Morphological Processing & Segmentation**: Dilation, Erosion, Opening, Closing, Otsu's Thresholding, Connected Component Analysis.
6. **End-to-End Capstone Application**: Automated Defect & Feature Extraction on Surface Images.


In [ ]:
# 1. Environment Setup and Library Imports
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version: {np.__version__}")

# Configure plotting aesthetics
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['image.cmap'] = 'gray'


In [ ]:
# 2. Test Image Generation / Loading
def get_sample_image():
    # Create a synthetic benchmark test pattern containing gradients, edges, text, and noise
    img = np.zeros((300, 300), dtype=np.uint8)
    # Background gradient
    for i in range(300):
        img[i, :] = int(i * 0.4)
    # Geometric shapes
    cv2.circle(img, (80, 80), 45, 220, -1)
    cv2.rectangle(img, (160, 40), (260, 140), 180, -1)
    # Fine details and text
    cv2.putText(img, "NINU IP 2026", (30, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255, 2)
    # Add salt and pepper noise patch
    noise = np.random.choice([0, 255], size=(300, 300), p=[0.97, 0.03]).astype(np.uint8)
    noisy_img = cv2.bitwise_or(img, noise)
    return noisy_img

sample_gray = get_sample_image()
sample_rgb = cv2.cvtColor(sample_gray, cv2.COLOR_GRAY2RGB)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(sample_gray, cmap='gray')
plt.title("Synthetic Benchmark Image (Grayscale)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.hist(sample_gray.ravel(), 256, [0, 256], color='steelblue')
plt.title("Image Intensity Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.show()


---
## 🌈 Part 1: Color Spaces & Intensity Point Transformations
Transforming pixel intensities directly using mapping functions $s = T(r)$:
- **Negative Transformation**: $s = L - 1 - r$
- **Power-Law (Gamma) Correction**: $s = c \cdot r^\gamma$
- **Logarithmic Transformation**: $s = c \cdot \log(1 + r)$


In [ ]:
# 3. Point Transformations
def apply_point_transformations(image):
    # 1. Negative
    img_neg = 255 - image
    
    # 2. Gamma Correction (gamma < 1 expands dark regions; gamma > 1 darkens)
    gamma_low = np.array(255 * (image / 255.0) ** 0.5, dtype='uint8')
    gamma_high = np.array(255 * (image / 255.0) ** 2.0, dtype='uint8')
    
    # 3. Log Transform
    c = 255 / np.log(1 + np.max(image))
    log_transformed = np.array(c * (np.log(1 + image.astype(np.float32))), dtype=np.uint8)
    
    return img_neg, gamma_low, gamma_high, log_transformed

neg, g_low, g_high, log_img = apply_point_transformations(sample_gray)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(neg, cmap='gray'); axes[0].set_title("Negative ($s = 255 - r$)"); axes[0].axis('off')
axes[1].imshow(g_low, cmap='gray'); axes[1].set_title(r"Gamma Correction ($\gamma=0.5$)"); axes[1].axis('off')
axes[2].imshow(g_high, cmap='gray'); axes[2].set_title(r"Gamma Correction ($\gamma=2.0$)"); axes[2].axis('off')
axes[3].imshow(log_img, cmap='gray'); axes[3].set_title("Log Transform"); axes[3].axis('off')
plt.tight_layout()
plt.show()


---
## 📊 Part 2: Histogram Equalization & Contrast Enhancement
Enhancing global and local dynamic range:
- **Global Histogram Equalization (HE)**
- **Contrast Limited Adaptive Histogram Equalization (CLAHE)**


In [ ]:
# 4. Global HE vs. CLAHE
def compare_hist_equalization(image):
    # Global Histogram Equalization
    global_he = cv2.equalizeHist(image)
    
    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    clahe_res = clahe.apply(image)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes[0, 0].imshow(image, cmap='gray'); axes[0, 0].set_title("Original Image"); axes[0, 0].axis('off')
    axes[0, 1].imshow(global_he, cmap='gray'); axes[0, 1].set_title("Global Equalization"); axes[0, 1].axis('off')
    axes[0, 2].imshow(clahe_res, cmap='gray'); axes[0, 2].set_title("CLAHE"); axes[0, 2].axis('off')
    
    axes[1, 0].hist(image.ravel(), 256, [0, 256], color='gray'); axes[1, 0].set_title("Original Histogram")
    axes[1, 1].hist(global_he.ravel(), 256, [0, 256], color='blue'); axes[1, 1].set_title("Global HE Histogram")
    axes[1, 2].hist(clahe_res.ravel(), 256, [0, 256], color='green'); axes[1, 2].set_title("CLAHE Histogram")
    plt.tight_layout()
    plt.show()
    return global_he, clahe_res

he_img, clahe_img = compare_hist_equalization(sample_gray)


---
## 🔍 Part 3: Spatial Filtering & Convolutions
Spatial domain neighborhood operations:
- Smoothing filters: Box/Averaging, Gaussian Filter, Median Filter (non-linear).
- Gradient and Sharpening operators: Sobel ($G_x, G_y$), Laplacian ($
abla^2 f$), and Canny Edge Detector.


In [ ]:
# 5. Spatial Domain Filtering Suite
def spatial_filtering_demo(image):
    # Smoothing
    blur_box = cv2.blur(image, (5, 5))
    blur_gauss = cv2.GaussianBlur(image, (5, 5), 1.5)
    filter_median = cv2.medianBlur(image, 5)
    
    # Edges & Gradients
    sobelx = cv2.Sobel(blur_gauss, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(blur_gauss, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.hypot(sobelx, sobely)
    sobel_mag = np.uint8(255 * sobel_mag / np.max(sobel_mag))
    
    laplacian = cv2.Laplacian(blur_gauss, cv2.CV_64F)
    laplacian_8u = cv2.convertScaleAbs(laplacian)
    
    canny = cv2.Canny(blur_gauss, 50, 150)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes[0, 0].imshow(blur_box, cmap='gray'); axes[0, 0].set_title("5x5 Box Mean Filter"); axes[0, 0].axis('off')
    axes[0, 1].imshow(blur_gauss, cmap='gray'); axes[0, 1].set_title("Gaussian Filter ($\sigma=1.5$)"); axes[0, 1].axis('off')
    axes[0, 2].imshow(filter_median, cmap='gray'); axes[0, 2].set_title("Median Filter (Noise Suppressed)"); axes[0, 2].axis('off')
    
    axes[1, 0].imshow(sobel_mag, cmap='gray'); axes[1, 0].set_title("Sobel Gradient Magnitude"); axes[1, 0].axis('off')
    axes[1, 1].imshow(laplacian_8u, cmap='gray'); axes[1, 1].set_title("Laplacian ($
abla^2 f$)"); axes[1, 1].axis('off')
    axes[1, 2].imshow(canny, cmap='gray'); axes[1, 2].set_title("Canny Multi-stage Edge Detector"); axes[1, 2].axis('off')
    plt.tight_layout()
    plt.show()

spatial_filtering_demo(sample_gray)


---
## 🌊 Part 4: Frequency Domain Image Processing (2D FFT)
Transforming spatial image $f(x, y)$ into frequency components $F(u, v)$ via 2D Discrete Fourier Transform.
Filtering is performed by multiplication in the frequency domain:
$$G(u, v) = H(u, v) \cdot F(u, v)$$


In [ ]:
# 6. Frequency Domain Filtering (Butterworth Low-Pass & High-Pass)
def frequency_domain_filtering(image, d0=30, n=2):
    # 2D FFT & Shift DC component to center
    f = np.fft.fft2(image)
    fshift = np.fft.fftshift(f)
    magnitude_spectrum = 20 * np.log(np.abs(fshift) + 1)
    
    rows, cols = image.shape
    crow, ccol = rows // 2, cols // 2
    
    # Distance matrix D(u, v)
    u = np.arange(rows)
    v = np.arange(cols)
    V, U = np.meshgrid(v, u)
    D = np.sqrt((U - crow)**2 + (V - ccol)**2)
    
    # Butterworth Low-Pass Filter: H(u, v) = 1 / (1 + (D / D0)^(2n))
    H_low = 1.0 / (1.0 + (D / d0)**(2 * n))
    
    # Butterworth High-Pass Filter: H(u, v) = 1 / (1 + (D0 / D)^(2n))
    D_safe = np.where(D == 0, 1e-6, D)
    H_high = 1.0 / (1.0 + (d0 / D_safe)**(2 * n))
    
    # Apply filters in frequency domain
    fshift_low = fshift * H_low
    fshift_high = fshift * H_high
    
    # Inverse FFT back to spatial domain
    img_low = np.abs(np.fft.ifft2(np.fft.ifftshift(fshift_low)))
    img_high = np.abs(np.fft.ifft2(np.fft.ifftshift(fshift_high)))
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(magnitude_spectrum, cmap='inferno'); axes[0].set_title("Centered FFT Magnitude"); axes[0].axis('off')
    axes[1].imshow(H_low, cmap='gray'); axes[1].set_title(f"Butterworth LPF ($D_0={d0}$)"); axes[1].axis('off')
    axes[2].imshow(img_low, cmap='gray'); axes[2].set_title("LPF Filtered (Smoothed)"); axes[2].axis('off')
    axes[3].imshow(img_high, cmap='gray'); axes[3].set_title("HPF Filtered (Edges)"); axes[3].axis('off')
    plt.tight_layout()
    plt.show()

frequency_domain_filtering(sample_gray, d0=40, n=2)


---
## 🧱 Part 5: Morphological Operations & Image Segmentation
- **Mathematical Morphology**: Structuring elements, Erosion, Dilation, Opening (removes small objects), Closing (fills small holes).
- **Thresholding**: Global fixed, Otsu's optimal clustering, and Connected Component Analysis.


In [ ]:
# 7. Morphology and Otsu Segmentation
def morphological_segmentation_demo(image):
    # Denoise first with median
    denoised = cv2.medianBlur(image, 5)
    
    # Otsu automatic optimal thresholding
    thresh_val, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    print(f"Otsu's Computed Optimal Threshold: {thresh_val:.2f}")
    
    # Morphological Structuring Element
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    
    erosion = cv2.erode(binary, kernel, iterations=1)
    dilation = cv2.dilate(binary, kernel, iterations=1)
    opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    closing = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    
    # Connected Components Labeling
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(opening)
    print(f"Detected Connected Component Objects: {num_labels - 1}")
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes[0, 0].imshow(binary, cmap='gray'); axes[0, 0].set_title("Otsu's Binary Mask"); axes[0, 0].axis('off')
    axes[0, 1].imshow(erosion, cmap='gray'); axes[0, 1].set_title("Erosion (Thinned)"); axes[0, 1].axis('off')
    axes[0, 2].imshow(dilation, cmap='gray'); axes[0, 2].set_title("Dilation (Expanded)"); axes[0, 2].axis('off')
    
    axes[1, 0].imshow(opening, cmap='gray'); axes[1, 0].set_title("Opening (Noise Removed)"); axes[1, 0].axis('off')
    axes[1, 1].imshow(closing, cmap='gray'); axes[1, 1].set_title("Closing (Holes Filled)"); axes[1, 1].axis('off')
    axes[1, 2].imshow(labels, cmap='tab20'); axes[1, 2].set_title("Connected Components Map"); axes[1, 2].axis('off')
    plt.tight_layout()
    plt.show()

morphological_segmentation_demo(sample_gray)


---
## 🎯 Part 6: Capstone Pipeline - Automated Feature Extraction & Defect Profiling
Combines all previous stages into a unified, production-grade inspection pipeline:
Preprocess $\to$ Noise Filter $\to$ Adaptive Threshold $\to$ Contour Analysis $\to$ Bounding Boxes & Object Measurements.


In [ ]:
# 8. End-to-End Object Inspection Pipeline
def run_inspection_pipeline(image):
    # 1. Preprocess
    blur = cv2.GaussianBlur(image, (5, 5), 0)
    _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 2. Morphological clean
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    clean_mask = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # 3. Contour detection
    contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    annotated = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    measurements = []
    
    for i, c in enumerate(contours):
        area = cv2.contourArea(c)
        if area > 100:  # filter noise
            perimeter = cv2.arcLength(c, True)
            x, y, w, h = cv2.boundingRect(c)
            # Circularity metric: 4 * pi * Area / (Perimeter^2)
            circularity = (4 * np.pi * area) / (perimeter**2) if perimeter > 0 else 0
            
            cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(annotated, f"#{i+1} A:{int(area)}", (x, max(15, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1)
            
            measurements.append({
                "Object_ID": i + 1,
                "Bounding_Box": (x, y, w, h),
                "Area_px": area,
                "Perimeter_px": round(perimeter, 2),
                "Circularity": round(circularity, 3)
            })
            
    print(f"Total Significant Objects Profiled: {len(measurements)}")
    for m in measurements:
        print(f"Object #{m['Object_ID']} -> Area: {m['Area_px']} px | Perimeter: {m['Perimeter_px']} px | Circularity: {m['Circularity']}")
        
    plt.figure(figsize=(8, 6))
    plt.imshow(annotated)
    plt.title("Automated Detection & Dimensional Profiling")
    plt.axis('off')
    plt.show()

run_inspection_pipeline(sample_gray)


---
## 🏁 Conclusion
- **Spatial filtering** provides rapid localized contrast and edge features.
- **Frequency domain processing** offers clean mathematical attenuation of periodic noise and high-frequency structural elements.
- **Morphological post-processing** bridges low-level pixel filtering and high-level object semantics.
